# 03 — EDA & Business Analysis

Goal: answer portfolio-relevant business questions from the cleaned source-of-truth dataset. All comparisons are descriptive.

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "boston_airbnb_cleaned.csv"
df = pd.read_csv(DATA_PATH)
df.shape

## Core KPIs

In [ ]:
kpis = pd.Series({
    "total_listings": len(df),
    "avg_price_usd": df["price_usd"].mean(),
    "median_price_usd": df["price_usd"].median(),
    "avg_rating": df["review_scores_rating"].mean(),
    "rated_listings": df["review_scores_rating"].notna().sum(),
    "avg_availability_365": df["availability_365"].mean(),
    "total_reviews": df["number_of_reviews"].sum(),
})
kpis

## 1. What is the listing mix by room type?

In [ ]:
room_type_summary = df.groupby("room_type").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median"),
    avg_rating=("review_scores_rating", "mean"),
    avg_availability_365=("availability_365", "mean")
).sort_values("listings", ascending=False)
room_type_summary["share_pct"] = room_type_summary["listings"] / len(df) * 100
room_type_summary

## 2. Which neighbourhoods contain the most listings?

In [ ]:
neighbourhood_summary = df.groupby("neighbourhood_cleansed").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median"),
    avg_rating=("review_scores_rating", "mean")
).sort_values("listings", ascending=False)
neighbourhood_summary.head(10)

## 3. Which sufficiently represented neighbourhoods have the highest median prices?

A 20-listing threshold is used only for this ranking to avoid interpreting tiny groups as stable neighbourhood-level comparisons. No rows are deleted from the dataset.

In [ ]:
neighbourhood_summary.query("listings >= 20").sort_values(["median_price_usd", "listings"], ascending=[False, False]).head(10)

## 4. How does property type differ in volume and price?

In [ ]:
df.groupby("property_type", dropna=False).agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median")
).sort_values("listings", ascending=False)

## 5. How does guest capacity relate descriptively to price?

In [ ]:
df.groupby("accommodates").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median")
).sort_index()

## 6. How do superhost and non-superhost listings compare?

In [ ]:
df.groupby("host_is_superhost").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median"),
    rated_listings=("review_scores_rating", "count"),
    avg_rating=("review_scores_rating", "mean"),
    avg_reviews=("number_of_reviews", "mean"),
    avg_availability_365=("availability_365", "mean")
)

## 7. How do instant-bookable listings compare?

In [ ]:
df.groupby("instant_bookable").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median"),
    rated_listings=("review_scores_rating", "count"),
    avg_rating=("review_scores_rating", "mean"),
    avg_reviews=("number_of_reviews", "mean")
)

## 8. What is the cancellation-policy mix?

In [ ]:
df.groupby("cancellation_policy").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median"),
    avg_rating=("review_scores_rating", "mean")
).sort_values("listings", ascending=False)

## 9. How concentrated are listings among hosts?

In [ ]:
host_summary = df.groupby("host_id").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean")
).sort_values("listings", ascending=False)
print("Unique hosts:", len(host_summary))
print("Hosts with one listing (%):", round(host_summary["listings"].eq(1).mean()*100, 2))
host_summary.head(10)

## 10. How complete are the review metrics?

In [ ]:
review_fields = [c for c in df.columns if c.startswith("review_scores_")]
pd.DataFrame({
    "available": df[review_fields].notna().sum(),
    "missing": df[review_fields].isna().sum(),
    "coverage_pct": df[review_fields].notna().mean()*100
}).sort_values("coverage_pct", ascending=False)

## Interpretation guardrails

- Results refer to the 2016 snapshot only.
- Missing review values are excluded from rating means.
- Availability is not treated as occupancy.
- Group differences are descriptive and should not be described as causal effects.
- Extreme nightly prices remain in the source-of-truth dataset.